# Stock Price Prediction with PyTorch — LSTM vs GRU

This notebook implements the requirements of the provided one-month stock-price-prediction project plan.

## Objective

Build a regression-based time-series model that predicts the next trading day's closing price using:

- historical stock-price data
- Pandas and scikit-learn for preparation
- PyTorch
- an LSTM model
- a GRU model
- MSE and RMSE evaluation
- a direct LSTM vs GRU comparison

The notebook intentionally keeps the project scope to the requirements in the plan: **one stock and next-day prediction**.

> This is a learning/portfolio project, not a trading system. Stock-price prediction is difficult and the results should not be interpreted as financial advice.


## 1. Imports and configuration

The notebook uses Yahoo Finance through `yfinance` for the historical stock dataset. The default example is Amazon (AMZN), matching the project plan's example.

The experiment uses a chronological train/test split because this is time-series data. **The scaler is fitted only on the training portion** to avoid test-data leakage.


In [ ]:
# If needed, install dependencies in your environment before running:
# pip install yfinance pandas numpy matplotlib scikit-learn torch

import time
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf

import torch
import torch.nn as nn
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

TICKER = "AMZN"
YEARS = 10
LOOKBACK = 20
TRAIN_RATIO = 0.80

HIDDEN_SIZE = 32
NUM_LAYERS = 2
DROPOUT = 0.20
EPOCHS = 50
LEARNING_RATE = 0.001

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("PyTorch:", torch.__version__)
print("Device:", DEVICE)
print("Ticker:", TICKER)


## 2. Load historical stock data

We download approximately 10 years of daily data and use the **Close** price as the single feature, as specified by the project plan.

The final project should use real historical data. If the download fails, the notebook stops with an informative error rather than silently generating synthetic stock data.


In [ ]:
end_date = pd.Timestamp.today().normalize()
start_date = end_date - pd.DateOffset(years=YEARS)

data = yf.download(
    TICKER,
    start=start_date.strftime("%Y-%m-%d"),
    end=(end_date + pd.Timedelta(days=1)).strftime("%Y-%m-%d"),
    progress=False,
    auto_adjust=True,
    threads=False
)

if data.empty:
    raise RuntimeError(
        f"No real Yahoo Finance data was returned for {TICKER}. "
        "Check your internet connection/ticker and rerun the notebook."
    )

# yfinance can return a MultiIndex depending on its version.
if isinstance(data.columns, pd.MultiIndex):
    if ("Close", TICKER) in data.columns:
        close = data[("Close", TICKER)]
    else:
        close = data["Close"].iloc[:, 0]
else:
    close = data["Close"]

close = close.dropna().astype(float)
close.name = "Close"

if len(close) < 100:
    raise RuntimeError("Too few observations were downloaded for a meaningful experiment.")

print(f"Downloaded {len(close):,} daily observations.")
print(f"Period: {close.index.min().date()} to {close.index.max().date()}")
close.head()


## 3. Exploratory data analysis

The project plan asks for basic exploration and a closing-price visualization. We also inspect descriptive statistics and missing values.


In [ ]:
print("Descriptive statistics:")
display(close.describe().to_frame())

print("Missing values:", close.isna().sum())
print("Number of observations:", len(close))


In [ ]:
plt.figure(figsize=(12, 5))
plt.plot(close.index, close.values)
plt.title(f"{TICKER} Closing Price — Historical Data")
plt.xlabel("Date")
plt.ylabel("Price")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


## 4. Train/test split

For a time series, the split is chronological:

- first 80% → training data
- final 20% → unseen test data

No future observations are used to train the model.

The scaler is fitted **only on the training prices**, which prevents information from the test period from influencing preprocessing.


In [ ]:
split_idx = int(len(close) * TRAIN_RATIO)

train_prices = close.iloc[:split_idx].to_numpy().reshape(-1, 1)
test_prices = close.iloc[split_idx:].to_numpy().reshape(-1, 1)

scaler = MinMaxScaler(feature_range=(-1, 1))
train_scaled = scaler.fit_transform(train_prices)

# Transform the test set using parameters learned only from training data.
test_scaled = scaler.transform(test_prices)

print("Training observations:", len(train_prices))
print("Test observations:", len(test_prices))
print("Training period:", close.index[:split_idx].min().date(), "to", close.index[:split_idx-1].date())
print("Test period:", close.index[split_idx:].min().date(), "to", close.index[-1].date())


## 5. Create sliding-window sequences

The model receives the previous `LOOKBACK` trading days and predicts the next trading day's closing price.

For example, with a lookback of 20:

`days 1–20 → predict day 21`

`days 2–21 → predict day 22`

and so on.


In [ ]:
def create_sequences(values, lookback):
    X, y = [], []
    for i in range(lookback, len(values)):
        X.append(values[i - lookback:i])
        y.append(values[i])
    return np.asarray(X, dtype=np.float32), np.asarray(y, dtype=np.float32).reshape(-1, 1)

X_train_np, y_train_np = create_sequences(train_scaled, LOOKBACK)

# Include the final LOOKBACK training observations before the test period
# so the first test prediction can use the immediately preceding history.
combined_for_test = np.vstack([train_scaled[-LOOKBACK:], test_scaled])
X_test_np, y_test_np = create_sequences(combined_for_test, LOOKBACK)

print("X_train:", X_train_np.shape)
print("y_train:", y_train_np.shape)
print("X_test:", X_test_np.shape)
print("y_test:", y_test_np.shape)


## 6. Convert data to PyTorch tensors

PyTorch models operate on tensors. The LSTM/GRU input shape is:

`[samples, sequence_length, features]`

Here there is one feature: the closing price.


In [ ]:
X_train = torch.from_numpy(X_train_np).to(DEVICE)
y_train = torch.from_numpy(y_train_np).to(DEVICE)
X_test = torch.from_numpy(X_test_np).to(DEVICE)
y_test = torch.from_numpy(y_test_np).to(DEVICE)

print("Tensor shapes:")
print("X_train:", X_train.shape)
print("y_train:", y_train.shape)
print("X_test:", X_test.shape)
print("y_test:", y_test.shape)


## 7. Define the LSTM and GRU models

Both models use the same general architecture and hyperparameters so that their results can be compared fairly.

- input size = 1
- hidden size = 32
- 2 recurrent layers
- dropout = 0.20 between recurrent layers
- output size = 1

The LSTM maintains hidden and cell states, while the GRU uses a simpler gated recurrent structure.


In [ ]:
# Quick shape check
with torch.no_grad():
    test_batch = X_train[:2]
    lstm_check = LSTMModel(
        input_size=1, hidden_size=HIDDEN_SIZE, num_layers=NUM_LAYERS,
        dropout=DROPOUT, output_size=1
    ).to(DEVICE)
    gru_check = GRUModel(
        input_size=1, hidden_size=HIDDEN_SIZE, num_layers=NUM_LAYERS,
        dropout=DROPOUT, output_size=1
    ).to(DEVICE)

    print("LSTM output shape:", lstm_check(test_batch).shape)
    print("GRU output shape:", gru_check(test_batch).shape)


## 8. Training function

The training loop uses:

- Mean Squared Error (`nn.MSELoss`) as the regression loss
- Adam as the optimizer
- forward pass
- loss calculation
- backpropagation
- optimizer update

Training loss is recorded so that convergence can be inspected.


In [ ]:
def train_model(model, X, y, epochs=50, learning_rate=0.001):
    model = model.to(DEVICE)
    criterion = nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

    losses = []
    start = time.perf_counter()

    model.train()
    for epoch in range(epochs):
        optimizer.zero_grad()

        predictions = model(X)
        loss = criterion(predictions, y)

        loss.backward()
        optimizer.step()

        losses.append(loss.item())

        if (epoch + 1) % 10 == 0 or epoch == 0:
            print(f"Epoch {epoch + 1:3d}/{epochs} | Training MSE: {loss.item():.6f}")

    elapsed = time.perf_counter() - start
    return model, losses, elapsed


## 9. Train the LSTM


In [ ]:
torch.manual_seed(SEED)

lstm_model = LSTMModel(
    input_size=1,
    hidden_size=HIDDEN_SIZE,
    num_layers=NUM_LAYERS,
    dropout=DROPOUT,
    output_size=1
).to(DEVICE)

lstm_model, lstm_losses, lstm_time = train_model(
    lstm_model, X_train, y_train,
    epochs=EPOCHS,
    learning_rate=LEARNING_RATE
)

print(f"LSTM training time: {lstm_time:.2f} seconds")


## 10. Train the GRU


In [ ]:
torch.manual_seed(SEED)

gru_model = GRUModel(
    input_size=1,
    hidden_size=HIDDEN_SIZE,
    num_layers=NUM_LAYERS,
    dropout=DROPOUT,
    output_size=1
).to(DEVICE)

gru_model, gru_losses, gru_time = train_model(
    gru_model, X_train, y_train,
    epochs=EPOCHS,
    learning_rate=LEARNING_RATE
)

print(f"GRU training time: {gru_time:.2f} seconds")


## 11. Compare training loss

The training-loss curves help identify whether the models learned during training and whether the loss generally decreased.


In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(lstm_losses, label="LSTM")
plt.plot(gru_losses, label="GRU")
plt.title("Training MSE")
plt.xlabel("Epoch")
plt.ylabel("MSE")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


## 12. Evaluate on unseen test data

Predictions are generated only for the test set.

MSE and RMSE are calculated after converting predictions back to the original price scale, making RMSE interpretable in the same units as the stock price.


In [ ]:
def predict_model(model, X):
    model.eval()
    with torch.no_grad():
        return model(X).detach().cpu().numpy()

lstm_pred_scaled = predict_model(lstm_model, X_test)
gru_pred_scaled = predict_model(gru_model, X_test)

y_test_original = scaler.inverse_transform(y_test_np)
lstm_pred_original = scaler.inverse_transform(lstm_pred_scaled)
gru_pred_original = scaler.inverse_transform(gru_pred_scaled)

lstm_mse = mean_squared_error(y_test_original, lstm_pred_original)
gru_mse = mean_squared_error(y_test_original, gru_pred_original)

lstm_rmse = np.sqrt(lstm_mse)
gru_rmse = np.sqrt(gru_mse)

print(f"LSTM MSE:  {lstm_mse:.4f}")
print(f"LSTM RMSE: {lstm_rmse:.4f}")
print(f"GRU MSE:   {gru_mse:.4f}")
print(f"GRU RMSE:  {gru_rmse:.4f}")


## 13. Actual vs predicted prices

The plots compare the actual unseen test prices with each model's predictions. This provides a visual complement to the numerical MSE/RMSE metrics.


In [ ]:
test_dates = close.index[split_idx:]

# The number of test predictions equals the number of test observations.
assert len(test_dates) == len(y_test_original)

plt.figure(figsize=(12, 5))
plt.plot(test_dates, y_test_original.ravel(), label="Actual")
plt.plot(test_dates, lstm_pred_original.ravel(), label="LSTM Predicted")
plt.title(f"{TICKER} — LSTM Test Predictions")
plt.xlabel("Date")
plt.ylabel("Price")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(12, 5))
plt.plot(test_dates, y_test_original.ravel(), label="Actual")
plt.plot(test_dates, gru_pred_original.ravel(), label="GRU Predicted")
plt.title(f"{TICKER} — GRU Test Predictions")
plt.xlabel("Date")
plt.ylabel("Price")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


## 14. LSTM vs GRU comparison

The project requires a direct comparison of model performance and training time.

Lower MSE/RMSE means lower prediction error on this test set. Training time is reported for the same number of epochs and the same general architecture/hyperparameters.


In [ ]:
results = pd.DataFrame({
    "Model": ["LSTM", "GRU"],
    "MSE": [lstm_mse, gru_mse],
    "RMSE": [lstm_rmse, gru_rmse],
    "Training Time (seconds)": [lstm_time, gru_time]
})

display(results)

best_model = results.loc[results["RMSE"].idxmin(), "Model"]
fastest_model = results.loc[results["Training Time (seconds)"].idxmin(), "Model"]

print(f"Lower test RMSE: {best_model}")
print(f"Faster training in this run: {fastest_model}")


## 15. Optional moderate hyperparameter experiment

The project plan says moderate tuning may be performed **if needed**. This small experiment changes the lookback window while keeping the architecture fixed.

This is not intended to be an exhaustive search. It demonstrates the iterative process without turning the project into a large optimization study.

Importantly, each experiment still fits its scaler only on the training portion.


In [ ]:
# A small, optional tuning experiment.
# Set RUN_TUNING = False if you only want the required main experiment.
RUN_TUNING = True

if RUN_TUNING:
    tuning_rows = []

    for lookback in [10, 20]:
        X_tune_train_np, y_tune_train_np = create_sequences(train_scaled, lookback)
        combined_tune = np.vstack([train_scaled[-lookback:], test_scaled])
        X_tune_test_np, y_tune_test_np = create_sequences(combined_tune, lookback)

        X_tune_train = torch.from_numpy(X_tune_train_np).to(DEVICE)
        y_tune_train = torch.from_numpy(y_tune_train_np).to(DEVICE)
        X_tune_test = torch.from_numpy(X_tune_test_np).to(DEVICE)

        torch.manual_seed(SEED)
        tune_model = GRUModel(
            input_size=1,
            hidden_size=HIDDEN_SIZE,
            num_layers=NUM_LAYERS,
            dropout=DROPOUT,
            output_size=1
        ).to(DEVICE)

        tune_model, _, tune_time = train_model(
            tune_model,
            X_tune_train,
            y_tune_train,
            epochs=30,
            learning_rate=LEARNING_RATE
        )

        tune_pred_scaled = predict_model(tune_model, X_tune_test)
        tune_actual = scaler.inverse_transform(y_tune_test_np)
        tune_pred = scaler.inverse_transform(tune_pred_scaled)

        tune_mse = mean_squared_error(tune_actual, tune_pred)
        tune_rmse = np.sqrt(tune_mse)

        tuning_rows.append({
            "Lookback": lookback,
            "GRU Test MSE": tune_mse,
            "GRU Test RMSE": tune_rmse,
            "Training Time (seconds)": tune_time
        })

    tuning_results = pd.DataFrame(tuning_rows)
    display(tuning_results)
else:
    print("Hyperparameter tuning skipped.")


## 16. Findings and limitations

### What to report

Use the results table above to state which model achieved the lower test MSE/RMSE and which model trained faster in this experiment. Do not assume that one architecture will always win.

### Limitations

- The experiment uses only the closing price as the input feature.
- Historical patterns do not guarantee future stock-price behavior.
- The model is evaluated on one historical test period.
- The experiment is intended for learning, not trading decisions.
- More features, alternative architectures, and longer/more robust validation could be explored in future work.

### Next steps

Possible extensions include using additional market features, testing other lookback windows, trying more advanced time-series models, and evaluating the approach across multiple stocks.


## 17. Final checklist

This notebook contains the core deliverables specified by the project plan:

- [x] Historical stock data
- [x] Pandas data handling
- [x] Exploratory price visualization
- [x] Preprocessing and normalization
- [x] Sliding-window sequences
- [x] Chronological train/test split
- [x] PyTorch tensors
- [x] LSTM model
- [x] GRU model
- [x] MSE loss
- [x] Adam optimizer
- [x] Training loops
- [x] Test-set predictions
- [x] MSE and RMSE
- [x] Training-time comparison
- [x] Actual vs predicted plots
- [x] LSTM vs GRU comparison
- [x] Explanation of limitations and next steps

The separate `README.md` in this project documents setup, usage, dataset, and results.
